# Treinamento do detector YOLO (objeto único) — táticas do repositório

Notebook **só de YOLO**: treina o detector que localiza "o objeto", **sem distinguir MILCO/NOMBO** (classe única
`object`), usando as táticas de `MODELS/TESTES_RECORTES_YOLO_CONVNEXT/yolo_gpu/treinar_yolo_gpu.py` em vez de vários
scripts soltos. Não há classificador (ConvNeXt/GhostNet/MobileNet) nem CBAM aqui.

## O que veio do repositório

| Tática | Onde está neste notebook |
|---|---|
| **Classe única** `object` (`single_cls=True`, rótulos = 0) — o tipo é decidido depois por outro modelo | Seção 3 e 5 |
| **Split determinístico** (arquivos ordenados por nome, `Random(42).shuffle`, 70/15/15 → 819/175/176) | Seção 3 |
| **Variantes G1–G6**: YOLOv8n-P2 640 (=Modelo A), 1024 px, YOLOv8s-P2, YOLO11s, **G5** = augmentação p/ objetos pequenos, **G6** = AdamW | Seção 5 |
| G5/G6 partem da **melhor de G2–G4 escolhida na validação** | Seção 5 |
| Treino sem mosaico e **sem `flipud`** (a sombra acústica do sonar tem direção fixa), `patience=50`, `seed=42`, teto de tempo (`time`), cache em RAM opcional | Seção 5 |
| **Retomável**: `FEITO.json` pula o que terminou; `last.pt` retoma o que foi interrompido | Seção 5 |
| **Avaliador próprio**: AP50, recall/precisão em conf 0,25, FP por imagem, **recall de objetos pequenos** (<32² px) | Seção 4 |
| **Escolha pela validação**: `AP50 + recall de pequenos` (o teste só reporta) | Seção 6 |
| (Opcional) **TTA, blocos 2×2 e fusão WBF** de dois detectores, ponto de operação por F1/F2 na validação | Seção 8 |

## Saída (mesma estrutura do repositório)

`saida/pesos/G*_best.pt`, `saida/runs/G*/` (curvas e `FEITO.json`), `saida/resultados_yolo_gpu.csv`,
`saida/resumo.json` e `saida/detector_config.json`. O CSV tem as mesmas colunas lidas por
`fase_e_detector_gpu.py` do repositório.

> **Tempo:** no repositório, G1–G6 levaram ~7 h em 2× T4 (60–180 min por variante). Use `HORAS_POR_VARIANTE` e
> `VARIANTES_A_TREINAR` (Seção 2) para ajustar. O notebook é retomável: se o kernel cair, rode de novo.

## 1. Imports

In [1]:
# !pip install "ultralytics>=8.3" pandas matplotlib pillow
# (instale antes o PyTorch com CUDA: https://pytorch.org/get-started/locally/)
import os, gc, json, csv, random, shutil, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
import ultralytics
from ultralytics import YOLO

pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
print("Ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (CPU)")

Ultralytics 8.4.163 | torch 2.14.0+cu126
GPU: NVIDIA GeForce RTX 3050


## 2. Configuração

**Dataset.** `ORIGEM_DATASET = "existente"` reutiliza o `yolo_dataset` que você já tem (o mesmo split do notebook
original: 819/175/176 imagens). Com `"raw"`, o notebook refaz o dataset a partir das imagens + `.txt` originais
(pasta extraída ou zip do figshare), com a mesma regra de split do repositório.

**Variantes.** Os valores abaixo são os do repositório. Para uma GPU de 8 GB (ex.: RTX 3050) os modelos de 1024 px
podem estourar memória: o notebook **reduz o batch pela metade automaticamente** se ocorrer `out of memory`.

In [2]:
# ----------------------------------------------------------------------------- pastas
PROJETO_DIR = Path("D:/testte/data-science-notebooks/MODELS/TESTE_YOLO_NEW")     # <-- saída deste notebook
SAIDA = PROJETO_DIR / "saida"      # pesos/, runs/, resultados_yolo_gpu.csv, resumo.json, detector_config.json
WORK  = PROJETO_DIR / "work"       # dataset gerado (modo "raw") e arquivos temporários

ORIGEM_DATASET = "existente"       # "existente" | "raw"
DATASET_EXISTENTE = Path("D:/testte/SEPARACAO_MODELS/TESTE_BOX_YOLO/work/yolo_dataset")   # images/{train,val,test} + labels/...
RAW_ORIGEM = "D:/testte/SEPARACAO_MODELS/raw_data"   # modo "raw": pasta com .jpg+.txt, pasta com zips por ano ou zip único
ESPERADO_TOTAL = 1170              # nº de imagens do SIMD (modo "raw"); None = não checar

# ----------------------------------------------------------------------------- o que treinar
VARIANTES_A_TREINAR = ["G1", "G2", "G3", "G4", "G5", "G6"]   # ex.: ["G1", "G3", "G5"] para encurtar
EPOCHS = 200                       # só vale se HORAS_POR_VARIANTE = 0 (com teto de tempo, o ultralytics ignora epochs)
HORAS_POR_VARIANTE = 2.0           # teto de tempo de treino por variante (h); 0 = sem teto
BATCH = 0                          # 0 = regra do repositório (16 se imgsz<=640, senão 8)
BATCH_POR_VARIANTE = {}            # ex.: {"G3": 4}
CACHE_RAM = False                  # True acelera, mas consome RAM (seu kernel já caiu por RAM antes)
WORKERS = 0 if os.name == "nt" else 4   # Windows + Jupyter: 0 evita travamentos do DataLoader
SMOKE = False                      # True = teste rápido (1 época, 320 px, 10% dos dados)

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------------- variantes (idênticas ao repositório)
VARIANTES = {
    "G1": dict(modelo="yolov8n-p2.yaml", base="yolov8n.pt", imgsz=640),     # = Modelo A da apresentação
    "G2": dict(modelo="yolov8n-p2.yaml", base="yolov8n.pt", imgsz=1024),
    "G3": dict(modelo="yolov8s-p2.yaml", base="yolov8s.pt", imgsz=1024),
    "G4": dict(modelo="yolo11s.pt", base=None, imgsz=1024),
    # G5/G6 = melhor de G2-G4 (escolhida na VALIDAÇÃO) + ajuste extra:
    "G5": dict(derivada=True, extra=dict(mosaic=0.3, copy_paste=0.3, scale=0.3, flipud=0.0)),   # objetos pequenos
    "G6": dict(derivada=True, extra=dict(optimizer="AdamW", lr0=0.001)),
}
BASES_DERIVADAS = ("G2", "G3", "G4")

# pesos de referência já treinados por você (opcional, Seção 7)
PESOS_REFERENCIA = {
    "A existente (YOLOv8n-P2 640)": Path("D:/testte/SEPARACAO_MODELS/TESTE_BOX_YOLO/runs/detect/work/runs/modelA_yolo_p2/weights/best.pt"),
}

# ----------------------------------------------------------------------------- táticas de inferência (opcional, Seção 8)
RODAR_TATICAS_INFERENCIA = False   # TTA, blocos 2x2 e fusão WBF de dois detectores (só inferência, sem novo treino)

for d in (SAIDA, WORK):
    d.mkdir(parents=True, exist_ok=True)
(SAIDA / "pesos").mkdir(exist_ok=True)
print("Saída:", SAIDA)

Saída: D:\testte\data-science-notebooks\MODELS\TESTE_YOLO_NEW\saida


## 3. Dataset em classe única

O YOLO só precisa saber **onde há um objeto** — o tipo (MILCO × NOMBO) é decisão de outro modelo. Por isso:

- o `data.yaml` usado aqui (`saida/data_object.yaml`) tem uma única classe `object`;
- o treino usa `single_cls=True`, então mesmo que seu `yolo_dataset` tenha rótulos de 2 classes, eles viram 1;
- **não altero os arquivos do seu dataset** (o ultralytics só grava os `.cache` de rótulos dentro dele, como já fazia).

In [3]:
IMG_EXT = {".jpg", ".jpeg", ".png"}
SPLITS = ("train", "val", "test")


# ---------------------------------------------------------------- split e dataset a partir dos originais (modo "raw")
def dividir_imagens(imgs):
    """Split do repositório / notebook original: ordenado por nome, Random(42).shuffle, 70/15/15."""
    sh = sorted(imgs, key=os.path.basename)
    random.Random(SEED).shuffle(sh)
    n = len(sh)
    ntr, nva = int(n * 0.70), int(n * 0.15)
    return {"train": sh[:ntr], "val": sh[ntr:ntr + nva], "test": sh[ntr + nva:]}


def _extrair_zips(origem, destino):
    origem = Path(origem)
    if destino.exists():
        return
    if origem.is_file():                      # zip único "baixar tudo" do figshare
        tmp, apagar = destino.parent / "_zip", True
        with zipfile.ZipFile(origem) as z:
            z.extractall(tmp)
        base = tmp
    else:                                     # pasta com os zips por ano
        base, apagar = origem, False
    for zp in sorted(base.glob("*.zip")):
        with zipfile.ZipFile(zp) as z:
            z.extractall(destino)
    if apagar:
        shutil.rmtree(base)


def preparar_dataset_raw(origem, work):
    origem = Path(origem)
    if origem.is_dir() and next(origem.rglob("*.jpg"), None) is not None:
        imgs = [str(p) for p in origem.rglob("*.jpg")]            # já extraído
    else:
        raw = work / "raw"
        _extrair_zips(origem, raw)
        imgs = [str(p) for p in raw.rglob("*.jpg")]
    if ESPERADO_TOTAL:
        assert len(imgs) == ESPERADO_TOTAL, f"esperava {ESPERADO_TOTAL} imagens, achei {len(imgs)}"
    splits = dividir_imagens(imgs)
    ds = work / "yolo_dataset"
    for s, lst in splits.items():
        (ds / "images" / s).mkdir(parents=True, exist_ok=True)
        (ds / "labels" / s).mkdir(parents=True, exist_ok=True)
        for p in lst:
            stem = Path(p).stem
            dst = ds / "images" / s / f"{stem}.jpg"
            if not dst.exists():
                shutil.copy(p, dst)
            lab = ds / "labels" / s / f"{stem}.txt"
            if lab.exists():
                continue
            linhas = []
            txt = Path(p).with_suffix(".txt")
            if txt.exists():
                for l in txt.read_text().splitlines():
                    q = l.split()
                    if len(q) >= 5:
                        linhas.append("0 " + " ".join(q[1:5]))    # classe única; coords já YOLO (centro, normalizado)
            lab.write_text("\n".join(linhas))                      # arquivo vazio = imagem de fundo
    json.dump({k: [Path(p).name for p in v] for k, v in splits.items()}, open(work / "split.json", "w"), indent=1)
    return ds


# ---------------------------------------------------------------- leitura de imagens/rótulos (usada pelo avaliador)
def listar_imagens(ds, split):
    d = Path(ds) / "images" / split
    return sorted([p for p in d.iterdir() if p.suffix.lower() in IMG_EXT], key=lambda p: p.name)


def carregar_gts(ds, split):
    """{stem: [[x1,y1,x2,y2] em pixels]} — a classe do rótulo é ignorada (objeto único)."""
    gts = {}
    for img in listar_imagens(ds, split):
        w, h = Image.open(img).size
        lab = Path(ds) / "labels" / split / f"{img.stem}.txt"
        g = []
        if lab.exists():                                  # sem arquivo = imagem de fundo
            for l in lab.read_text().splitlines():
                q = l.split()
                if len(q) == 5:
                    xc, yc, bw, bh = map(float, q[1:])
                    g.append([(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h])
        gts[img.stem] = g
    return gts


def escrever_yaml(ds, destino):
    destino.write_text(f"path: {Path(ds).resolve().as_posix()}\ntrain: images/train\nval: images/val\n"
                       f"test: images/test\nnames:\n  0: object\n")


# ---------------------------------------------------------------- escolhe/gera o dataset
if ORIGEM_DATASET == "existente":
    DS = Path(DATASET_EXISTENTE)
    assert (DS / "images").exists(), f"Não encontrei {DS / 'images'}. Ajuste DATASET_EXISTENTE."
    for s in SPLITS:
        assert (DS / "images" / s).exists(), f"Falta {DS / 'images' / s}"
else:
    DS = preparar_dataset_raw(RAW_ORIGEM, WORK)

DATA_YAML = SAIDA / "data_object.yaml"
escrever_yaml(DS, DATA_YAML)
print("Dataset:", DS)
print("data.yaml (classe única):", DATA_YAML)

# ---------------------------------------------------------------- estatísticas
GTS = {s: carregar_gts(DS, s) for s in SPLITS}
classes_originais = set()
for s in SPLITS:
    for lab in (DS / "labels" / s).glob("*.txt"):
        for l in lab.read_text().splitlines():
            q = l.split()
            if len(q) == 5:
                classes_originais.add(q[0])
print("Classes encontradas nos rótulos originais:", sorted(classes_originais), "-> todas viram 'object'")

linhas = []
for s in SPLITS:
    g = GTS[s]
    areas = [(b[2] - b[0]) * (b[3] - b[1]) for v in g.values() for b in v]
    linhas.append({"split": s, "imagens": len(g), "com_objeto": sum(len(v) > 0 for v in g.values()),
                   "fundo": sum(len(v) == 0 for v in g.values()), "objetos": len(areas),
                   "pequenos(<32²px)": sum(a < 32 * 32 for a in areas)})
ESTAT = pd.DataFrame(linhas).set_index("split")
display(ESTAT)
esperado = {"train": 819, "val": 175, "test": 176}
if {s: int(ESTAT.loc[s, "imagens"]) for s in SPLITS} != esperado:
    print("AVISO: o split difere do repositório (819/175/176). Os resultados não serão diretamente comparáveis.")

Dataset: D:\testte\SEPARACAO_MODELS\TESTE_BOX_YOLO\work\yolo_dataset
data.yaml (classe única): D:\testte\data-science-notebooks\MODELS\TESTE_YOLO_NEW\saida\data_object.yaml
Classes encontradas nos rótulos originais: ['0'] -> todas viram 'object'


,imagens,com_objeto,fundo,objetos,pequenos(<32²px)
split,,,,,
train,819,217,602,484,304
val,175,41,134,75,53
test,176,46,130,109,68


## 4. Avaliador de detecção (porta do `avaliar` do repositório)

Independente do ultralytics. Para classe única, com IoU 0,5:

- **AP50** (todos os pontos da curva precisão×recall);
- **recall, precisão e falsos positivos por imagem** no conf operacional (0,25);
- **recall de objetos pequenos** (área < 32² px) — o critério que o repositório soma ao AP50 para escolher o detector.

As predições são coletadas com `conf=0.001` (curva completa). Imagens sem objeto contam para os falsos positivos.

In [4]:
def iou_xyxy(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def nms_np(b, iou_thr):
    """NMS guloso por confiança. b: [N,5] = x1,y1,x2,y2,conf."""
    if len(b) == 0:
        return b
    ordem = np.argsort(-b[:, 4])
    manter = []
    for i in ordem:
        if all(iou_xyxy(b[i, :4], b[j, :4]) <= iou_thr for j in manter):
            manter.append(i)
    return b[manter]


def _predizer(model, caminho, imgsz, device, augment=False, iou=0.6, tiles=0):
    """Caixas [N,5] (x1,y1,x2,y2,conf) com conf >= 0.001. tiles=k: k×k blocos (sobreposição 0,2) + imagem inteira."""
    def _um(entrada, desloc=(0, 0), aug=False):
        r = model.predict(entrada, imgsz=imgsz, conf=0.001, iou=iou, augment=aug, device=device, verbose=False)[0]
        if len(r.boxes) == 0:
            return np.zeros((0, 5))
        b = r.boxes.xyxy.cpu().numpy() + np.array([desloc[0], desloc[1], desloc[0], desloc[1]])
        return np.concatenate([b, r.boxes.conf.cpu().numpy()[:, None]], 1)

    if not tiles:
        return _um(caminho, aug=augment)
    im = Image.open(caminho).convert("RGB")
    W, H = im.size
    ov = 0.2
    tw, th = W / (tiles - (tiles - 1) * ov), H / (tiles - (tiles - 1) * ov)
    out = []
    for i in range(tiles):
        for j in range(tiles):
            x0, y0 = i * tw * (1 - ov), j * th * (1 - ov)
            out.append(_um(im.crop((int(x0), int(y0), int(x0 + tw), int(y0 + th))), (int(x0), int(y0))))
    out.append(_um(caminho))                                        # + imagem inteira
    out = [o for o in out if len(o)]
    return np.concatenate(out) if out else np.zeros((0, 5))


def coletar_deteccoes(model, split, imgsz, device=None, augment=False, iou=0.6, tiles=0):
    device = DEVICE if device is None else device
    return {img.stem: _predizer(model, str(img), imgsz, device, augment, iou, tiles) for img in listar_imagens(DS, split)}


def metricas_deteccao(dets, gts, conf_op=0.25, iou_nms=None):
    """dets: {stem: [N,5]}; gts: {stem: [[x1,y1,x2,y2]]}. iou_nms=None -> usa as caixas como vieram."""
    n_gt = n_small = tp_small = tp_op = fp_op = 0
    pares = []
    for stem, g in gts.items():
        n_gt += len(g)
        d = dets.get(stem)
        d = np.zeros((0, 5)) if d is None else d
        if iou_nms is not None:
            d = nms_np(d, iou_nms)
        d = d[np.argsort(-d[:, 4])] if len(d) else d
        usado, usado_op = set(), set()
        for b in d:
            j = max(range(len(g)), key=lambda k: iou_xyxy(b[:4], g[k]), default=None)
            ok = j is not None and j not in usado and iou_xyxy(b[:4], g[j]) >= 0.5
            if ok:
                usado.add(j)
            pares.append((b[4], ok))
            if b[4] >= conf_op:
                if ok:
                    tp_op += 1; usado_op.add(j)
                else:
                    fp_op += 1
        for k, gb in enumerate(g):
            if (gb[2] - gb[0]) * (gb[3] - gb[1]) < 32 * 32:
                n_small += 1
                tp_small += k in usado_op
    pares.sort(key=lambda t: -t[0])
    oks = np.array([o for _, o in pares], dtype=float)
    ctp, cfp = np.cumsum(oks), np.cumsum(1 - oks)
    rec, prec = ctp / max(n_gt, 1), ctp / np.maximum(ctp + cfp, 1e-9)
    mrec, mpre = np.r_[0, rec, 1], np.r_[1, prec, 0]
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    return {"AP50": float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:])), "conf": conf_op,
            "recall": tp_op / max(n_gt, 1), "precisao": tp_op / max(tp_op + fp_op, 1),
            "FP_por_img": fp_op / max(len(gts), 1), "recall_pequenos": tp_small / max(n_small, 1),
            "n_objetos": n_gt, "n_pequenos": n_small}


def avaliar_modelo(model, split, imgsz, conf_op=0.25):
    """Formato do repositório (colunas do resultados_yolo_gpu.csv)."""
    m = metricas_deteccao(coletar_deteccoes(model, split, imgsz), GTS[split], conf_op)
    return {"split": split, "AP50": m["AP50"], f"recall@{conf_op:g}": m["recall"],
            f"precisao@{conf_op:g}": m["precisao"], f"FP_por_imagem@{conf_op:g}": m["FP_por_img"],
            f"recall_pequenos@{conf_op:g}": m["recall_pequenos"], "n_objetos": m["n_objetos"], "n_pequenos": m["n_pequenos"]}


def limpar_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Treino das variantes (receita do `treinar_yolo_gpu.py`)

Parâmetros fixos do repositório: `single_cls=True`, `mosaic=0.0`, `flipud=0.0`, `patience=50`, `seed=42`,
batch 16 (≤640 px) ou 8 (1024 px), 200 épocas com teto de tempo opcional. Os pesos iniciais vêm do COCO
(`yolov8n.pt`/`yolov8s.pt` carregados no modelo P2; `yolo11s.pt` direto).

- **G5** acrescenta `mosaic=0.3, copy_paste=0.3, scale=0.3` (objetos pequenos). *(O `copy_paste` usa máscaras de
  segmentação; em dados só com caixas costuma não ter efeito — mantido por fidelidade ao repositório.)*
- **G6** troca o otimizador para `AdamW` com `lr0=0.001`.
- G5/G6 usam como base a **melhor de G2–G4 na validação** (critério `AP50 + recall de pequenos`).

**Retomada:** variante com `FEITO.json` é pulada; variante interrompida continua do `last.pt`.

In [5]:
def criterio_val(r):
    """Critério de escolha do repositório (na VALIDAÇÃO): AP50 + recall de objetos pequenos."""
    return r["val"]["AP50"] + r["val"]["recall_pequenos@0.25"]


def resolver_cfg(nome, resultados):
    cfg = dict(VARIANTES[nome])
    if not cfg.get("derivada"):
        return cfg
    base = []
    for g in BASES_DERIVADAS:
        f = SAIDA / "runs" / g / "FEITO.json"
        r = resultados.get(g) or (json.load(open(f)) if f.exists() else None)
        if r:
            base.append(r)
    if not base:
        print(f"{nome}: precisa de pelo menos uma entre {BASES_DERIVADAS} antes; pulando")
        return None
    melhor = max(base, key=criterio_val)
    print(f"{nome}: baseada em {melhor['variante']} (melhor na validação entre {[b['variante'] for b in base]})")
    return dict(VARIANTES[melhor["variante"]], extra=VARIANTES[nome]["extra"])


def _eh_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def treinar_variante(nome, cfg):
    runs = SAIDA / "runs"
    feito = runs / nome / "FEITO.json"
    if feito.exists():
        print(f"[{nome}] já concluída (FEITO.json) - pulando")
        return json.load(open(feito))

    last, best = runs / nome / "weights" / "last.pt", runs / nome / "weights" / "best.pt"
    imgsz = 320 if SMOKE else cfg["imgsz"]
    batch = BATCH_POR_VARIANTE.get(nome) or BATCH or (16 if cfg["imgsz"] <= 640 else 8)
    t0 = time.time()

    while True:
        kw = dict(data=str(DATA_YAML), epochs=1 if SMOKE else EPOCHS, imgsz=imgsz, batch=batch, device=DEVICE,
                  single_cls=True, mosaic=0.0, flipud=0.0, patience=50, project=str(runs), name=nome,
                  exist_ok=True, seed=SEED, workers=0 if SMOKE else WORKERS, plots=not SMOKE)
        if HORAS_POR_VARIANTE > 0:
            kw["time"] = HORAS_POR_VARIANTE
        if CACHE_RAM:
            kw["cache"] = "ram"
        if SMOKE:
            kw["fraction"] = 0.1
        kw.update(cfg.get("extra", {}))
        model = None
        try:
            if last.exists() and not SMOKE:
                print(f"[{nome}] retomando de {last}")
                model = YOLO(str(last))
                model.train(resume=True)
            else:
                print(f"[{nome}] treinando: modelo={cfg['modelo']} imgsz={imgsz} batch={batch} extra={cfg.get('extra', {})}")
                model = YOLO(cfg["modelo"])
                if cfg.get("base"):
                    model.load(cfg["base"])          # pesos COCO nas camadas compatíveis
                model.train(**kw)
            break
        except Exception as e:
            msg = str(e).lower()
            if "nothing to resume" in msg or "is finished" in msg:
                print(f"[{nome}] o treino já havia terminado; seguindo para a avaliação")
                break
            if _eh_oom(e) and batch > 2 and not last.exists():
                batch //= 2
                print(f"[{nome}] sem memória de GPU -> tentando com batch={batch}")
                model = None
                limpar_gpu()
                continue
            raise
    del model
    limpar_gpu()

    assert best.exists(), f"best.pt não encontrado em {best} (o treino terminou sem checkpoint?)"
    modelo = YOLO(str(best))
    res = {"variante": nome, "modelo": cfg["modelo"], "imgsz": imgsz, "batch": batch, "extra": cfg.get("extra", {}),
           "treino_min": round((time.time() - t0) / 60, 1),
           "val": avaliar_modelo(modelo, "val", imgsz), "test": avaliar_modelo(modelo, "test", imgsz)}
    shutil.copy(best, SAIDA / "pesos" / f"{nome}_best.pt")
    json.dump(res, open(feito, "w"), indent=1)
    del modelo
    limpar_gpu()
    return res


def consolidar_csv():
    """resultados_yolo_gpu.csv no mesmo formato do repositório (lido por fase_e_detector_gpu.py)."""
    linhas = []
    for f in sorted((SAIDA / "runs").glob("G*/FEITO.json")):
        r = json.load(open(f))
        for sp in ("val", "test"):
            linhas.append({"variante": r["variante"], "modelo": r["modelo"], "imgsz": r["imgsz"],
                           "extra": json.dumps(r["extra"]), "treino_min": r["treino_min"], **r[sp]})
    if linhas:
        with open(SAIDA / "resultados_yolo_gpu.csv", "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(linhas[0].keys()))
            w.writeheader(); w.writerows(linhas)
    return linhas


resultados = {}
for nome in VARIANTES_A_TREINAR:
    cfg = resolver_cfg(nome, resultados)
    if cfg is None:
        continue
    print(f"\n=== {nome}: {cfg}")
    resultados[nome] = treinar_variante(nome, cfg)
    v = resultados[nome]["val"]
    print(f"[{nome}] validação: AP50 {v['AP50']:.3f} | recall@0.25 {v['recall@0.25']:.3f} | "
          f"recall pequenos {v['recall_pequenos@0.25']:.3f} | tempo {resultados[nome]['treino_min']} min")

linhas_csv = consolidar_csv()
print("\nCSV:", SAIDA / "resultados_yolo_gpu.csv")


=== G1: {'modelo': 'yolov8n-p2.yaml', 'base': 'yolov8n.pt', 'imgsz': 640}
[G1] treinando: modelo=yolov8n-p2.yaml imgsz=640 batch=16 extra={}
Transferred 219/437 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.192 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\testte\data-science-notebooks\MODELS\TESTE_YOLO_NEW\saida\data_object.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=Tru

C:\Users\Usuário\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 10.78.1 MB/s, size: 146.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\testte\SEPARACAO_MODELS\TESTE_BOX_YOLO\work\yolo_dataset\labels\val.cache... 175 images, 134 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 175/175  0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 70 weight(decay=0.0), 79 weight(decay=0.0005), 78 bias(decay=0.0)
Plotting labels to D:\testte\data-science-notebooks\MODELS\TESTE_YOLO_NEW\saida\runs\G1\labels.jpg... 
Using 81

KeyboardInterrupt: 

## 6. Resultados e escolha do detector (na validação)

O ranking usa **apenas a validação** (`AP50 + recall de pequenos`). As colunas de teste estão na tabela só para
reportar — com 109 objetos no teste, diferenças de poucos pontos entre variantes são ruído.

In [ ]:
runs_feitos = {}
for f in sorted((SAIDA / "runs").glob("G*/FEITO.json")):
    r = json.load(open(f)); runs_feitos[r["variante"]] = r
assert runs_feitos, "Nenhuma variante concluída ainda (rode a Seção 5)."

linhas = []
for nome, r in runs_feitos.items():
    linhas.append({"variante": nome, "modelo": r["modelo"], "imgsz": r["imgsz"], "extra": json.dumps(r["extra"]) if r["extra"] else "",
                   "min": r["treino_min"],
                   "val AP50": r["val"]["AP50"], "val rec.peq.": r["val"]["recall_pequenos@0.25"],
                   "CRITÉRIO (val)": criterio_val(r),
                   "val recall": r["val"]["recall@0.25"], "val prec.": r["val"]["precisao@0.25"],
                   "teste AP50": r["test"]["AP50"], "teste recall": r["test"]["recall@0.25"],
                   "teste prec.": r["test"]["precisao@0.25"], "teste FP/img": r["test"]["FP_por_imagem@0.25"],
                   "teste rec.peq.": r["test"]["recall_pequenos@0.25"]})
TAB = pd.DataFrame(linhas).set_index("variante").sort_values("CRITÉRIO (val)", ascending=False)
display(TAB)

MELHOR = TAB.index[0]
MELHOR_R = runs_feitos[MELHOR]
PESOS_MELHOR = SAIDA / "pesos" / f"{MELHOR}_best.pt"
print(f"\nMelhor na validação: {MELHOR} ({MELHOR_R['modelo']}, {MELHOR_R['imgsz']} px) -> {PESOS_MELHOR}")

# ---------------------------------------------------------------- gráficos
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
x = np.arange(len(TAB)); w = 0.38
axs[0].bar(x - w / 2, TAB["val AP50"], w, label="validação"); axs[0].bar(x + w / 2, TAB["teste AP50"], w, label="teste")
axs[0].set_title("AP50 (IoU 0,5)")
axs[1].bar(x - w / 2, TAB["val rec.peq."], w, label="validação"); axs[1].bar(x + w / 2, TAB["teste rec.peq."], w, label="teste")
axs[1].set_title("Recall de objetos pequenos (<32² px) @conf 0,25")
axs[2].bar(x - w / 2, TAB["val recall"], w, label="recall val"); axs[2].bar(x + w / 2, TAB["val prec."], w, label="precisão val")
axs[2].set_title("Recall × precisão (validação) @conf 0,25")
for a in axs:
    a.set_xticks(x); a.set_xticklabels(TAB.index); a.legend(); a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------- curvas de treino (results.csv do ultralytics)
fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
for nome in runs_feitos:
    f = SAIDA / "runs" / nome / "results.csv"
    if not f.exists():
        continue
    df = pd.read_csv(f); df.columns = df.columns.str.strip()
    if "metrics/mAP50(B)" in df:
        axs[0].plot(df["epoch"], df["metrics/mAP50(B)"], label=nome)
    if "val/box_loss" in df:
        axs[1].plot(df["epoch"], df["val/box_loss"], label=nome)
axs[0].set_title("mAP50 de validação por época (ultralytics)"); axs[1].set_title("Loss de caixa na validação")
for a in axs:
    a.set_xlabel("época"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

json.dump({"melhor_na_validacao": MELHOR, "criterio": "AP50 + recall_pequenos@0.25 (validação)",
           "variantes": {k: {"val": v["val"], "test": v["test"], "imgsz": v["imgsz"], "modelo": v["modelo"], "extra": v["extra"]}
                         for k, v in runs_feitos.items()}}, open(SAIDA / "resumo.json", "w"), indent=1)

## 7. (Opcional) Comparar com um modelo que você já treinou

Avalia pesos existentes (ex.: o seu Modelo A) com **o mesmo avaliador e o mesmo split**, para comparar com as
variantes. A variante `G1` é a reprodução do Modelo A com a receita do repositório.

In [ ]:
ref_linhas = []
for nome, p in PESOS_REFERENCIA.items():
    p = Path(p)
    if not p.exists():
        print(f"[ignorado] {nome}: {p} não encontrado"); continue
    m = YOLO(str(p))
    imgsz = int(getattr(m, "overrides", {}).get("imgsz", 640) or 640)
    v, t = avaliar_modelo(m, "val", imgsz), avaliar_modelo(m, "test", imgsz)
    ref_linhas.append({"modelo": nome, "imgsz": imgsz, "val AP50": v["AP50"], "val rec.peq.": v["recall_pequenos@0.25"],
                       "CRITÉRIO (val)": v["AP50"] + v["recall_pequenos@0.25"], "teste AP50": t["AP50"],
                       "teste recall": t["recall@0.25"], "teste prec.": t["precisao@0.25"]})
    del m; limpar_gpu()
if ref_linhas:
    comp = pd.concat([pd.DataFrame(ref_linhas).set_index("modelo"),
                      TAB.reset_index().assign(modelo=lambda d: d["variante"] + " (" + d["modelo"] + ")")
                         .set_index("modelo")[["imgsz", "val AP50", "val rec.peq.", "CRITÉRIO (val)", "teste AP50", "teste recall", "teste prec."]]])
    display(comp.sort_values("CRITÉRIO (val)", ascending=False))
else:
    print("Nenhum peso de referência encontrado.")

## 8. (Opcional) Táticas de inferência do repositório — sem novo treino

Ative com `RODAR_TATICAS_INFERENCIA = True` (Seção 2). Usa o melhor detector da Seção 6:

1. **Configurações de inferência:** padrão, **TTA** (`augment=True`) e **blocos 2×2** + imagem inteira. As predições
   são coletadas com `conf=0.001` e NMS mínimo (IoU 0,9); o **IoU do NMS** final é escolhido na validação (0,5/0,6/0,7).
2. **Fusão WBF** (Weighted Boxes Fusion, IoU 0,55) do melhor detector com TTA + um segundo YOLO com TTA (por padrão a
   `G1`, como "G5+TTA ⊕ Modelo A+TTA" do repositório).
3. **Ponto de operação** (conf) escolhido na validação por **F1** (equilibrado) e **F2** (recall pesa 4×).

No repositório, o TTA foi a melhor mudança de inferência (AP50 de validação 0,668 → 0,722 no Modelo A) e ampliar a
imagem na inferência **piorou** (o modelo foi treinado em um tamanho fixo).

In [ ]:
def wbf(listas, iou_thr=0.55):
    """Weighted Boxes Fusion (Solovyev et al., 2021), como em fusao_detectores.py. listas: [N,5] por modelo."""
    M = len(listas)
    caixas = [(b, k) for k, L in enumerate(listas) for b in L]
    caixas.sort(key=lambda t: -t[0][4])
    clusters = []
    for b, k in caixas:
        alvo = None
        for c in clusters:
            if iou_xyxy(b[:4], c["f"][:4]) > iou_thr:
                alvo = c; break
        if alvo is None:
            clusters.append({"m": [(b, k)], "f": b.copy()})
        else:
            alvo["m"].append((b, k))
            B = np.array([x for x, _ in alvo["m"]]); w = B[:, 4:5]
            alvo["f"] = np.r_[(B[:, :4] * w).sum(0) / w.sum(), B[:, 4].mean()]
    out = []
    for c in clusters:
        n_mod = len({k for _, k in c["m"]})
        f = c["f"].copy(); f[4] = f[4] * min(n_mod, M) / M      # penaliza caixas vistas por poucos modelos
        out.append(f)
    return np.array(out) if out else np.zeros((0, 5))


def fbeta(m, beta):
    p, r = m["precisao"], m["recall"]
    return (1 + beta ** 2) * p * r / max(beta ** 2 * p + r, 1e-9)


def escolher_operacao(dets_val, iou_nms, beta=1.0, grade=np.arange(0.05, 0.65, 0.05)):
    """Conf que maximiza F-beta NA VALIDAÇÃO (desempate por recall)."""
    melhor = None
    for c in grade:
        c = round(float(c), 2)
        m = metricas_deteccao(dets_val, GTS["val"], c, iou_nms)
        k = (fbeta(m, beta), m["recall"])
        if melhor is None or k > melhor[0]:
            melhor = (k, c, m)
    return melhor[1], melhor[2]


def tabela_config(nome, dets):
    """Escolhe o IoU do NMS na validação e devolve (linha_da_tabela, iou_escolhido)."""
    iou = max((0.5, 0.6, 0.7), key=lambda i: metricas_deteccao(dets["val"], GTS["val"], 0.25, i)["AP50"])
    v, t = (metricas_deteccao(dets[s], GTS[s], 0.25, iou) for s in ("val", "test"))
    return {"config": nome, "NMS": iou, "val AP50": v["AP50"], "val recall": v["recall"], "val prec.": v["precisao"],
            "val rec.peq.": v["recall_pequenos"], "teste AP50": t["AP50"], "teste recall": t["recall"],
            "teste prec.": t["precisao"], "teste FP/img": t["FP_por_img"]}, iou


TATICA_ESCOLHIDA = None
if RODAR_TATICAS_INFERENCIA:
    imgsz = MELHOR_R["imgsz"]
    modelo = YOLO(str(PESOS_MELHOR))
    cfgs = {f"{MELHOR} padrão": {}, f"{MELHOR} + TTA": dict(augment=True), f"{MELHOR} blocos 2x2": dict(tiles=2)}
    DETS, linhas, ious = {}, [], {}
    for nome, kw in cfgs.items():
        DETS[nome] = {s: coletar_deteccoes(modelo, s, imgsz, iou=0.9, **kw) for s in ("val", "test")}
        linha, ious[nome] = tabela_config(nome, DETS[nome])
        linhas.append(linha)
    del modelo; limpar_gpu()

    # ---- fusão WBF: melhor detector com TTA + segundo detector com TTA
    segundo = "G1" if (MELHOR != "G1" and "G1" in runs_feitos) else next((v for v in TAB.index if v != MELHOR), None)
    if segundo:
        m2 = YOLO(str(SAIDA / "pesos" / f"{segundo}_best.pt"))
        d2 = {s: coletar_deteccoes(m2, s, runs_feitos[segundo]["imgsz"], iou=0.9, augment=True) for s in ("val", "test")}
        del m2; limpar_gpu()
        d1 = DETS[f"{MELHOR} + TTA"]
        nome_f = f"WBF: {MELHOR}+TTA ⊕ {segundo}+TTA"
        DETS[nome_f] = {s: {k: wbf([nms_np(d1[s][k], 0.5), nms_np(d2[s][k], 0.5)]) for k in d1[s]} for s in ("val", "test")}
        linha, ious[nome_f] = tabela_config(nome_f, DETS[nome_f])
        linhas.append(linha)
    else:
        print("Só uma variante disponível: fusão WBF pulada.")

    TAB_TAT = pd.DataFrame(linhas).set_index("config")
    display(TAB_TAT)
    TATICA_ESCOLHIDA = TAB_TAT["val AP50"].idxmax()
    print(f"\nMelhor configuração na validação (AP50): {TATICA_ESCOLHIDA}")

    # ---- ponto de operação na validação (F1 e F2) para a configuração escolhida
    iou_e = ious[TATICA_ESCOLHIDA]
    for rotulo, beta in (("F1 (equilibrado)", 1.0), ("F2 (recall pesa 4x)", 2.0)):
        c, mv = escolher_operacao(DETS[TATICA_ESCOLHIDA]["val"], iou_e, beta)
        mt = metricas_deteccao(DETS[TATICA_ESCOLHIDA]["test"], GTS["test"], c, iou_e)
        print(f"{rotulo:20s} conf={c:.2f} | val: recall {mv['recall']:.3f} prec {mv['precisao']:.3f} FP/img {mv['FP_por_img']:.3f}"
              f" | teste: recall {mt['recall']:.3f} prec {mt['precisao']:.3f} FP/img {mt['FP_por_img']:.3f}")
else:
    print("RODAR_TATICAS_INFERENCIA = False (seção pulada)")

## 9. Exportar o detector escolhido

Grava `saida/detector_config.json` no formato do bloco `detector` dos `pipeline_config_*.json` do repositório:
pesos, tamanho de entrada, TTA (se a Seção 8 foi executada e escolheu uma configuração com TTA), `conf` e
`iou_nms` — todos escolhidos **na validação**. As métricas de teste são só informativas.

In [ ]:
imgsz = MELHOR_R["imgsz"]
modelo = YOLO(str(PESOS_MELHOR))
dets_v = coletar_deteccoes(modelo, "val", imgsz, iou=0.9)       # padrão (sem TTA), NMS refeito abaixo
dets_t = coletar_deteccoes(modelo, "test", imgsz, iou=0.9)
del modelo; limpar_gpu()

iou_nms = max((0.5, 0.6, 0.7), key=lambda i: metricas_deteccao(dets_v, GTS["val"], 0.25, i)["AP50"])
conf, mv = escolher_operacao(dets_v, iou_nms, beta=1.0)
mt = metricas_deteccao(dets_t, GTS["test"], conf, iou_nms)

cfg_det = {
    "descricao": f"Detector YOLO de objeto único ({MELHOR}: {MELHOR_R['modelo']}, {imgsz} px), treinado com a receita do repositório",
    "detector": {"pesos": str(PESOS_MELHOR.resolve().as_posix()), "imgsz": imgsz, "augment": False,
                 "conf": conf, "iou_nms": iou_nms},
    "criterio_escolha": "variante por AP50 + recall_pequenos (validação); conf por F1 (validação); iou_nms por AP50 (validação)",
    "metricas_validacao": {k: mv[k] for k in ("AP50", "recall", "precisao", "FP_por_img", "recall_pequenos")},
    "metricas_teste": {k: mt[k] for k in ("AP50", "recall", "precisao", "FP_por_img", "recall_pequenos")},
}
json.dump(cfg_det, open(SAIDA / "detector_config.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(cfg_det, indent=1, ensure_ascii=False))
if TATICA_ESCOLHIDA and TATICA_ESCOLHIDA != f"{MELHOR} padrão":
    print(f"\nObs.: na Seção 8, '{TATICA_ESCOLHIDA}' teve o melhor AP50 de validação. O arquivo acima usa a inferência "
          f"padrão; para usar TTA, defina \"augment\": true (e, para a fusão WBF, os dois pesos).")